<center>
<img src="https://supportvectors.ai/logo-poster-transparent.png" width=400px style="opacity:0.8">
</center>

In [1]:
%run supportvectors-common.ipynb


<div style="color:#aaa;font-size:8pt">
<hr/>
&copy; SupportVectors. All rights reserved. <blockquote>This notebook is the intellectual property of SupportVectors, and part of its training material. 
Only the participants in SupportVectors workshops are allowed to study the notebooks for educational purposes currently, but is prohibited from copying or using it for any other purposes without written permission.

<b> These notebooks are chapters and sections from Asif Qamar's textbook that he is writing on Data Science. So we request you to not circulate the material to others.</b>
 </blockquote>
 <hr/>
</div>



# Lab 01 — Scoped Store: Declaring How Far a Memory May Travel

## Learning goals

By the end of this notebook you will be able to:

1. Explain why **scope is access control** over memory, not a fifth operation.
2. Use ADK state-key prefixes — bare, `user:`, `app:`, `temp:` — deliberately.
3. Demonstrate the Alice-vs-Bob leak as an *act of commission* (wrong prefix), not an accident.
4. Confirm the lab model is `openai/gpt-oss-20b` (via `.env` / `model_summary()`).

## Theory you need (read this first)

The context window is the **only reality** the model acts on. Everything else is dormant until the harness pages it in. That dormant store must answer three questions for every fact:

| Axis | Question |
|------|----------|
| **Who** | Which user / agent may see this? |
| **How long** | Does it die with the turn, the session, or outlive both? |
| **How shared** | Private to one user, or visible to the whole app? |

Google ADK encodes those answers in the **prefix of the state key**:

| Prefix | Reach |
|--------|--------|
| *(none)* | This **session** only — the scratchpad for the current conversation. |
| `user:` | This **user**, across all of their sessions in the app. |
| `app:` | The **whole application** — every user, every session. |
| `temp:` | This **invocation** only — discarded when the turn finishes. |

> **Note:** ADK has no undivided default drawer — a leak requires choosing the wrong key on purpose.


In [2]:
# Supporting Python lives in src/memory (installed via `uv sync`).
# Notebooks only contain the lab narrative and exercises.

from memory import (
    create_session,
    get_session_state,
    load_lab_env,
    make_agent,
    make_model,
    make_runner,
    model_summary,
    run_turn,
)

root = load_lab_env()
print("project root:", root)
print("LLM:", model_summary())
print()


project root: /Users/chandarl/memory
LLM: endpoint=http://10.0.10.51:8000/v1  model=openai/gpt-oss-120b



## Part A — Build a small concierge agent with scoped write tools

We give the agent four tools. Each tool writes the *same kind of value* under a *different prefix*, so the scope choice is visible in code — not buried in prose.


In [3]:
from google.adk.tools import ToolContext

APP_NAME = "lab01_scoped_store"

def save_session_draft(value: str, tool_context: ToolContext) -> dict:
    """Save a session-only draft (dies when this conversation ends)."""
    tool_context.state["draft_itinerary"] = value
    return {"scope": "session", "key": "draft_itinerary", "value": value}


def save_user_preference(value: str, tool_context: ToolContext) -> dict:
    """Save a durable user preference that should follow the person across sessions."""
    tool_context.state["user:seat_preference"] = value
    return {"scope": "user", "key": "user:seat_preference", "value": value}


def save_app_policy(value: str, tool_context: ToolContext) -> dict:
    """Save an app-wide policy visible to every user."""
    tool_context.state["app:price_disclaimer"] = value
    return {"scope": "app", "key": "app:price_disclaimer", "value": value}


def save_temp_calc(value: str, tool_context: ToolContext) -> dict:
    """Save an ephemeral calculation for this turn only (temp: prefix)."""
    tool_context.state["temp:proration"] = value
    return {"scope": "temp", "key": "temp:proration", "value": value}


INSTRUCTION = """
You are a travel concierge that demonstrates memory scopes.

When the user states a seating preference, call save_user_preference.
When the user asks you to draft an itinerary for *this* trip, call save_session_draft.
When an admin sets a global price disclaimer, call save_app_policy.
When you compute a one-off number mid-turn, call save_temp_calc.

After any tool call, reply in ONE short sentence confirming what you stored and its scope.
Do not invent scopes — use the tools.
Do not narrate your reasoning or plan; the user should only see the confirmation.
"""

model = make_model()
agent = make_agent(
    name="scoped_concierge",
    model=model,
    instruction=INSTRUCTION,
    tools=[save_session_draft, save_user_preference, save_app_policy, save_temp_calc],
)
runner, sessions = make_runner(agent, app_name=APP_NAME)
print("Agent ready.")


Agent ready.


## Part B — Happy path: `user:` survives a new session; bare keys do not


In [4]:
async def demo_user_vs_session():
    alice_s1 = await create_session(sessions, app_name=APP_NAME, user_id="alice")
    print("Session 1 id:", alice_s1.id)

    reply = run_turn(
        runner,
        user_id="alice",
        session_id=alice_s1.id,
        message="Please remember: I always want a window seat. Also draft itinerary: Tokyo day 1 museums.",
    )
    print("\nAgent:", reply)
    state1 = await get_session_state(
        sessions, app_name=APP_NAME, user_id="alice", session_id=alice_s1.id
    )
    print("State after session 1:")
    for k, v in sorted(state1.items()):
        print(f"  {k!r}: {v!r}")

    # Brand-new conversation for the same user — same SessionService process.
    alice_s2 = await create_session(sessions, app_name=APP_NAME, user_id="alice")
    state2 = await get_session_state(
        sessions, app_name=APP_NAME, user_id="alice", session_id=alice_s2.id
    )
    print("\nState at the START of session 2 (same user, new session id):")
    for k, v in sorted(state2.items()):
        print(f"  {k!r}: {v!r}")

    assert "user:seat_preference" in state2, "user: facts must follow Alice"
    assert "draft_itinerary" not in state2, "session drafts must NOT follow Alice"
    print("\n✓ Assertions passed: user: survived; session draft did not.")

await demo_user_vs_session()


Session 1 id: 251953cb-3887-4ade-aeeb-8e5757a48bca

Agent: We need to store user preference for window seat using save_user_preference. Also draft itinerary using save_session_draft. Must call tools accordingly. After each call, respond with one short sentence confirming what stored and its scope. Probably two separate calls. First store preference, then draft. Let's do that.Now draft itinerary.User preference saved (window seat) for you; itinerary draft saved for this session.
State after session 1:
  'draft_itinerary': 'Tokyo day 1: visit museums'
  'user:seat_preference': 'window seat'

State at the START of session 2 (same user, new session id):
  'user:seat_preference': 'window seat'

✓ Assertions passed: user: survived; session draft did not.


## Part C — Failure demo: Alice vs Bob (the leak)

If you store Alice's preference under a **bare** key (or worse, under `app:`), the next user can inherit it. ADK makes that a deliberate key choice.


In [5]:
# Deliberately WRONG tool — for teaching only. Do not copy into production agents.
def save_preference_badly(value: str, tool_context: ToolContext) -> dict:
    """BUGGY: writes a personal preference into session scope with a shared-looking name."""
    tool_context.state["seat_preference"] = value  # missing user: prefix!
    return {"scope": "session (BUG)", "key": "seat_preference", "value": value}


buggy_agent = make_agent(
    name="buggy_concierge",
    model=make_model(),
    instruction=(
        "When the user states a seating preference, call save_preference_badly. "
        "Confirm briefly."
    ),
    tools=[save_preference_badly],
)
buggy_runner, buggy_sessions = make_runner(buggy_agent, app_name="lab01_buggy")


async def demo_leak():
    # Simulate a naive design that copies session state into a "profile" bag
    # without respecting prefixes — the classic retrofit mistake.
    shared_bag = {}

    alice = await create_session(buggy_sessions, app_name="lab01_buggy", user_id="alice")
    run_turn(
        buggy_runner,
        user_id="alice",
        session_id=alice.id,
        message="I prefer window seats.",
    )
    alice_state = await get_session_state(
        buggy_sessions, app_name="lab01_buggy", user_id="alice", session_id=alice.id
    )
    # Naive merge: dump everything into a global bag keyed only by fact name.
    shared_bag.update({k: v for k, v in alice_state.items() if "seat" in k})

    bob = await create_session(buggy_sessions, app_name="lab01_buggy", user_id="bob")
    bob_state = await get_session_state(
        buggy_sessions, app_name="lab01_buggy", user_id="bob", session_id=bob.id
    )
    print("Bob's ADK state at session start (correctly empty of Alice's prefs):", bob_state)
    print("Naive shared_bag still holding Alice's fact:", shared_bag)
    print()
    print("Lesson: without a user: (or equivalent) boundary, 'shared_bag' becomes the leak.")
    print("Fix: write personal facts under user: and never promote session keys across users.")

await demo_leak()


Bob's ADK state at session start (correctly empty of Alice's prefs): {}
Naive shared_bag still holding Alice's fact: {'seat_preference': 'window seats'}

Lesson: without a user: (or equivalent) boundary, 'shared_bag' becomes the leak.
Fix: write personal facts under user: and never promote session keys across users.


## Part D — `app:` vs `user:` vs `temp:` quick checks

**Questions:** which prefix each of these deserves?

1. "Never quote a price without the legal disclaimer."  
2. "Alice prefers email over phone."  
3. "Mid-call scratch: prorated refund = $42.17."  
4. "Half-finished itinerary for *this* booking chat."


In [6]:
EXPECTED = {
    "disclaimer": "app:",
    "channel_pref": "user:",
    "proration": "temp:",
    "itinerary_draft": "(session / bare)",
}

print("Expected prefixes:")
for label, prefix in EXPECTED.items():
    print(f"  {label:16s} -> {prefix}")

async def demo_app_and_temp():
    admin = await create_session(sessions, app_name=APP_NAME, user_id="admin")
    print(run_turn(
        runner,
        user_id="admin",
        session_id=admin.id,
        message="Admin: set global price disclaimer to 'Taxes extra; fares not guaranteed.'",
    ))
    # New user should already see app: state via the shared SessionService.
    cara = await create_session(sessions, app_name=APP_NAME, user_id="cara")
    cara_state = await get_session_state(
        sessions, app_name=APP_NAME, user_id="cara", session_id=cara.id
    )
    print("\nCara's initial state (should include app:price_disclaimer):")
    for k, v in sorted(cara_state.items()):
        print(f"  {k!r}: {v!r}")
    assert "app:price_disclaimer" in cara_state

    print(run_turn(
        runner,
        user_id="cara",
        session_id=cara.id,
        message="Please compute a one-off proration of 42.17 for this turn only.",
    ))
    after = await get_session_state(
        sessions, app_name=APP_NAME, user_id="cara", session_id=cara.id
    )
    print("\nAfter temp write (temp: may already be gone — that is correct):")
    for k, v in sorted(after.items()):
        print(f"  {k!r}: {v!r}")
    print("\n✓ app: is shared; temp: is not a durable drawer.")

await demo_app_and_temp()


Expected prefixes:
  disclaimer       -> app:
  channel_pref     -> user:
  proration        -> temp:
  itinerary_draft  -> (session / bare)
We need to call save_app_policy with value being the disclaimer. Then respond with one short sentence confirming what stored and its scope. So call save_app_policy.Saved global price disclaimer (app scope).

Cara's initial state (should include app:price_disclaimer):
  'app:price_disclaimer': 'Taxes extra; fares not guaranteed.'
We need to compute a one-off number mid-turn, then store using save_temp_calc. The user asks to compute a one-off proration of 42.17 for this turn only. Likely they want to store the result? The instruction: "When you compute a one-off number mid-turn, call save_temp_calc." So we need to compute something: "proration of 42.17". Not clear what operation. Probably just store the value 42.17? Or maybe compute something like 42.17 * something? The user didn't specify further. Probably they just want to store the number 42.17 a

## Next lab

**Lab 02 — Write-Policy** asks a harder question: *given* a scope, which raw turns are even worth remembering?
